# BenthiCat "SSS Pre-training": list, download a few sectors, inspect

Dataset: Harvard Dataverse `doi:10.7910/DVN/2VCN7Y` (about 1M unlabelled 384x384 side-scan sonar tiles, one 7z archive set per sector).

Run the cells in order. Do not download everything (about 54 GB for the sectors seen so far).
Send back the output of cell 2 and cell 4.

Edit `SECTORS` in cell 3 to change which sectors are fetched. Multi-part sectors (`.7z.001`, `.7z.002`, ...) need all parts downloaded; cell 3 does that.

## 1. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, requests, subprocess
DOI = "doi:10.7910/DVN/2VCN7Y"
BASE = "https://dataverse.harvard.edu"
ARCH = "/content/drive/MyDrive/benthicat/pretrain_archives"
LOCAL = "/content/bc_pre"
os.makedirs(ARCH, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
S = requests.Session()
S.headers.update({
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json, */*",
})

Mounted at /content/drive


## 2. List every file and size

In [3]:
def list_files():
    # Try 1: paginated files endpoint. Try 2: whole-dataset endpoint (files inside latestVersion).
    errs = []
    try:
        files, off = [], 0
        while True:
            r = S.get(f"{BASE}/api/datasets/:persistentId/versions/:latest/files",
                      params={"persistentId": DOI, "limit": 100, "offset": off}, timeout=60)
            r.raise_for_status()
            batch = r.json()["data"]
            if not batch: break
            files += batch; off += len(batch)
            if len(batch) < 100: break
        return files
    except Exception as e:
        errs.append(("files endpoint", e, getattr(getattr(e, "response", None), "text", "")[:300]))
    try:
        r = S.get(f"{BASE}/api/datasets/:persistentId/", params={"persistentId": DOI}, timeout=60)
        r.raise_for_status()
        return r.json()["data"]["latestVersion"]["files"]
    except Exception as e:
        errs.append(("dataset endpoint", e, getattr(getattr(e, "response", None), "text", "")[:300]))
    for name, e, body in errs:
        print("FAILED:", name, "->", e); print("   body:", body.replace("\n", " "))
    raise RuntimeError("Dataverse API listing failed, send me the lines above")

files = list_files()
rows = [(f.get("directoryLabel",""), f["label"], f["dataFile"]["id"], f["dataFile"]["filesize"]) for f in files]
for d, n, i, sz in sorted(rows):
    print(f"{d:5} {n:14} id={i:<9} {sz/1e6:9.1f} MB")
print(len(rows), "files,", sum(r[3] for r in rows)/1e9, "GB total")
json.dump(rows, open("/content/drive/MyDrive/benthicat/pretrain_index.json","w"))

N01   N01.7z.001     id=12510319     1178.8 MB
N02   N02.7z.001     id=12510334      148.0 MB
N03   N03.7z.001     id=12510338      672.2 MB
N04   N04.7z.001     id=12510337     2621.4 MB
N04   N04.7z.002     id=12510345     2621.4 MB
N04   N04.7z.003     id=12510328     1079.0 MB
N05   N05.7z.001     id=12510330     1443.9 MB
N06   N06.7z.001     id=12510321     1117.9 MB
N07   N07.7z.001     id=12510349     1839.0 MB
N08   N08.7z.001     id=12510342     1104.5 MB
N09   N09.7z.001     id=12510335     2490.9 MB
N10   N10.7z.001     id=12510327     1783.3 MB
N12   N12.7z.001     id=12510331     2621.4 MB
N12   N12.7z.002     id=12510323     2621.4 MB
N12   N12.7z.003     id=12510350     1112.7 MB
N15   N15.7z.001     id=12510320     2621.4 MB
N15   N15.7z.002     id=12510322      549.8 MB
N16   N16.7z.001     id=12510329     2621.4 MB
N16   N16.7z.002     id=12510339     2621.4 MB
N16   N16.7z.003     id=12510341      353.8 MB
S01   S01.7z.001     id=12510333     2621.4 MB
S01   S01.7z.

## 3. Download chosen sectors (resumes if interrupted)

In [4]:
SECTORS = ["N02", "N08", "N03"]
def fetch(fid, dest):
    have = os.path.getsize(dest) if os.path.exists(dest) else 0
    h = {"Range": f"bytes={have}-"} if have else {}
    with S.get(f"{BASE}/api/access/datafile/{fid}", headers=h, stream=True, timeout=120) as r:
        if r.status_code == 416: return
        r.raise_for_status()
        with open(dest, "ab" if have else "wb") as f:
            for chunk in r.iter_content(1<<20): f.write(chunk)
for d, n, fid, size in sorted(rows):
    if d in SECTORS:
        dest = os.path.join(ARCH, n)
        if os.path.exists(dest) and os.path.getsize(dest) == size:
            print("have", n); continue
        print("downloading", n, f"{size/1e6:.0f} MB"); fetch(fid, dest)
        assert os.path.getsize(dest) == size, f"size mismatch {n}"

downloading N02.7z.001 148 MB
downloading N03.7z.001 672 MB
downloading N08.7z.001 1104 MB


## 4. Extract and inspect

In [5]:
import glob, collections
from PIL import Image
if subprocess.run("which 7z || which 7zz", shell=True, capture_output=True).returncode != 0:
    subprocess.run("apt-get install -y -q p7zip-full", shell=True)
for s in SECTORS:
    first = os.path.join(ARCH, f"{s}.7z.001")
    out = os.path.join(LOCAL, s); os.makedirs(out, exist_ok=True)
    res = subprocess.run(["7z", "x", "-y", f"-o{out}", first], capture_output=True, text=True)
    if res.returncode != 0:
        print(s, "7z FAILED:", res.stdout[-500:], res.stderr[-500:])
        continue
    allf = [p for p in glob.glob(out + "/**/*", recursive=True) if os.path.isfile(p)]
    ext = collections.Counter(os.path.splitext(p)[1].lower() for p in allf)
    print(s, len(allf), "files", dict(ext))
    imgs = [p for p in allf if p.lower().endswith((".png",".jpg",".jpeg",".tif",".tiff"))][:5]
    for p in imgs:
        im = Image.open(p); print("  ", os.path.relpath(p, out), im.size, im.mode)

N02 2522 files {'.npy': 2522}
N08 17128 files {'.npy': 17128}
N03 10698 files {'.npy': 10698}


## 5. Inspect .npy tiles (shape, dtype, value range)
Send back the output of this cell too.

In [ ]:
import numpy as np
for s in SECTORS:
    fs = sorted(glob.glob(f"{LOCAL}/{s}/**/*.npy", recursive=True))
    if not fs: print(s, "no .npy found"); continue
    print(s, len(fs), "e.g.", os.path.relpath(fs[0], f"{LOCAL}/{s}"))
    for p in fs[:3] + fs[len(fs)//2:len(fs)//2+2]:
        a = np.load(p, allow_pickle=False)
        print("  ", os.path.basename(p), a.shape, a.dtype,
              "min/max/mean", float(a.min()), float(a.max()), round(float(a.mean()), 3),
              "nan:", bool(np.isnan(a).any()) if a.dtype.kind == "f" else "-")


## 6. SSL pretraining (SimCLR, ResNet-34)
Needs a GPU runtime (Runtime > Change runtime type > T4). Uses the sectors extracted in cell 4 (`SECTORS`, `LOCAL`).
Checkpoints go to Drive (`encoder.pt` and `last.pt` every 500 iterations). After a disconnect, re-run the same cell: it resumes from `last.pt` (and skips if a `DONE` marker exists).

In [ ]:
REPO = "/content/sonar-habitat-repro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/azmineamin/sonar-habitat-repro", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
OUT = "/content/drive/MyDrive/benthicat/ssl_r34_v224"
VAL_SECTORS = ["N03"]   # held out entirely (unseen-sector check); must be in SECTORS

Smoke test (about 1 minute). Check that loss prints and `encoder.pt` is written before the long run.

In [ ]:
!python {REPO}/seg/train_ssl.py --root {LOCAL} --sectors {" ".join(SECTORS)} --val-sectors {" ".join(VAL_SECTORS)} --iters 20 --batch 16 --workers 2 --eval-every 10 --out /content/ssl_smoke

Full run. 8000 iterations at batch 128 on 224x224 crops (full 384x384 measured about 1.75 s/iter on a T4, 20k iterations would take about 10 h; 224 should be about 3x faster, check the first log lines). This Colab machine has 2 CPUs, so `--workers 2`. Send back the last `eval` lines.

In [ ]:
!python {REPO}/seg/train_ssl.py --root {LOCAL} --sectors {" ".join(SECTORS)} --val-sectors {" ".join(VAL_SECTORS)} --iters 8000 --batch 128 --view-size 224 --workers 2 --eval-every 500 --out {OUT}

Re-score the finished run on the validation split (shuffled, all val tiles). Uses the same settings as the full run, and needs `last.pt` in `OUT`.

In [ ]:
!python {REPO}/seg/train_ssl.py --root {LOCAL} --sectors {" ".join(SECTORS)} --val-sectors {" ".join(VAL_SECTORS)} --iters 8000 --batch 128 --view-size 224 --workers 2 --eval-only --out {OUT}